In [0]:
WITH base_tours AS (
  SELECT DISTINCT
    dt.tour_id,
    o.tour_option_id,
    dt.tour_title,
    o.tour_option_title,
    dt.user_id AS supplier_id,
    dtl.location_name,
    dt.activity_type,
    dt.category AS tour_category,
    dt.date_of_creation,
    dt.date_of_first_online
  FROM production.dwh.dim_tour dt
  LEFT JOIN production.dwh.dim_tour_option o ON dt.tour_id = o.tour_id
  LEFT JOIN production.dwh.dim_tour_location dtl ON dtl.location_id = dt.location_id AND dtl.tour_id = dt.tour_id
  WHERE dt.gyg_status = 'active'
    AND dt.status = 'active'
    AND dt.is_online = true
    AND o.status = 'active'
    and tour_option_id IN (212949,318772,732424)
),

pricing_config_date_coverage AS (
  SELECT 
    tour_option_id,
    pricing_id,
    COUNT(DISTINCT CAST(date_time AS DATE)) AS num_dates_covered,
    MIN(CAST(date_time AS DATE)) AS earliest_date,
    MAX(CAST(date_time AS DATE)) AS latest_date
  FROM db_mirror_dbz.inventory_generated__available_time_slot
  WHERE CAST(date_time AS DATE) BETWEEN CURRENT_DATE() AND DATE_ADD(CURRENT_DATE(), 365)
  and tour_option_id IN (212949,318772,732424)
  GROUP BY tour_option_id, pricing_id
),

latest_config_per_pricing AS (
  SELECT 
    pcdc.tour_option_id,
    pcdc.pricing_id,
    pcdc.num_dates_covered,
    pcdc.earliest_date,
    pcdc.latest_date,
    ats.currency_iso_code,
    ats.details_deserialized,
    ats.update_timestamp,
    
    ROW_NUMBER() OVER (
      PARTITION BY pcdc.tour_option_id, pcdc.pricing_id
      ORDER BY ats.update_timestamp DESC
    ) as rn
    
  FROM pricing_config_date_coverage pcdc
  INNER JOIN db_mirror_dbz.inventory_generated__available_time_slot ats
    ON pcdc.tour_option_id = ats.tour_option_id 
    AND pcdc.pricing_id = ats.pricing_id
  WHERE ats.details_deserialized IS NOT NULL
),

pricing_config_exploded AS (
  SELECT 
    tour_option_id,
    pricing_id,
    currency_iso_code,
    num_dates_covered,
    earliest_date,
    latest_date,
    
    from_json(
      details_deserialized, 
      'struct<pricingCategories:struct<individual:array<struct<ticketCategory:string,minAge:int,maxAge:int,resolvedCommissionRate:decimal(10,3),independentlyBookable:boolean,bookingType:string,prices:array<struct<maxParticipants:int,retailPrice:decimal(10,2),netPrice:decimal(10,2)>>>>,group:array<struct<resolvedCommissionRate:decimal(10,3),isAdditionalPaxForGroup:boolean,prices:array<struct<maxParticipants:int,retailPrice:decimal(10,2),netPrice:decimal(10,2)>>>>,addons:array<struct<addonId:int,resolvedCommissionRate:decimal(10,3),prices:array<struct<maxParticipants:int,retailPrice:decimal(10,2),netPrice:decimal(10,2)>>>>>,group:array<struct<resolvedCommissionRate:decimal(10,3),isAdditionalPaxForGroup:boolean,prices:array<struct<maxParticipants:int,retailPrice:decimal(10,2),netPrice:decimal(10,2)>>>>,addons:array<struct<addonId:int,resolvedCommissionRate:decimal(10,3),prices:array<struct<maxParticipants:int,retailPrice:decimal(10,2),netPrice:decimal(10,2)>>>>>'
    ) as parsed_data,
    
    update_timestamp as config_snapshot_timestamp
    
  FROM latest_config_per_pricing
  WHERE rn = 1
),

normalized_pricing AS (
  SELECT
    tour_option_id,
    pricing_id,
    currency_iso_code,
    num_dates_covered,
    earliest_date,
    latest_date,
    
    COALESCE(parsed_data.pricingCategories.individual, ARRAY()) as individual_categories,
    
    COALESCE(
      parsed_data.pricingCategories.group,
      parsed_data.group
    ) as group_categories_raw,
    
    COALESCE(
      parsed_data.pricingCategories.addons,
      parsed_data.addons
    ) as addons_raw,
    
    config_snapshot_timestamp
    
  FROM pricing_config_exploded
),

pricing_features_per_config AS (
  SELECT 
    tour_option_id,
    pricing_id,
    currency_iso_code,
    num_dates_covered,
    earliest_date,
    latest_date,
    
    SIZE(individual_categories) as num_individual_categories,
    SIZE(group_categories_raw) as num_group_categories,
    SIZE(addons_raw) as num_addons,
    
    AGGREGATE(individual_categories, 0, (acc, ind) -> acc + SIZE(ind.prices)) as total_individual_price_tiers,
    AGGREGATE(group_categories_raw, 0, (acc, grp) -> acc + SIZE(grp.prices)) as total_group_price_tiers,
    AGGREGATE(addons_raw, 0, (acc, addon) -> acc + SIZE(addon.prices)) as total_addon_price_tiers,
    
    CASE 
      WHEN SIZE(FILTER(individual_categories, ind -> SIZE(ind.prices) > 1)) > 0 
      THEN 1 ELSE 0 
    END as has_volume_pricing_individual,
    
    CASE 
      WHEN SIZE(FILTER(group_categories_raw, grp -> SIZE(grp.prices) > 1)) > 0 
      THEN 1 ELSE 0 
    END as has_volume_pricing_group,
    
    CASE 
      WHEN SIZE(FILTER(addons_raw, addon -> SIZE(addon.prices) > 1)) > 0 
      THEN 1 ELSE 0 
    END as has_volume_pricing_addons,
    
    config_snapshot_timestamp
    
  FROM normalized_pricing
),

aggregated_features AS (
  SELECT
    tour_option_id,
    
    -- Take currency from the latest config
    MAX_BY(currency_iso_code, config_snapshot_timestamp) as currency_iso_code,
    
    -- Total distinct dates covered across all pricing_ids
    SUM(num_dates_covered) as pricing_dates_covered_next_12mo,
    
    -- Date range across all configs
    MIN(earliest_date) as pricing_valid_from,
    MAX(latest_date) as pricing_valid_to,
    
    -- MAX counts across all configs
    MAX(num_individual_categories) as num_individual_categories,
    MAX(num_group_categories) as num_group_categories,
    MAX(num_addons) as num_addons,
    
    -- MAX price tiers across all configs
    MAX(total_individual_price_tiers) as total_individual_price_tiers,
    MAX(total_group_price_tiers) as total_group_price_tiers,
    MAX(total_addon_price_tiers) as total_addon_price_tiers,
    
    -- Has feature if ANY config has it
    MAX(has_volume_pricing_individual) as has_volume_pricing_individual,
    MAX(has_volume_pricing_group) as has_volume_pricing_group,
    MAX(has_volume_pricing_addons) as has_volume_pricing_addons,
    
    -- Latest config timestamp
    MAX(config_snapshot_timestamp) as config_snapshot_timestamp
    
  FROM pricing_features_per_config
  GROUP BY tour_option_id
)

SELECT 
  bt.tour_id,
  bt.tour_option_id,
  bt.tour_title,
  bt.tour_option_title,
  bt.supplier_id,
  bt.location_name,
  bt.activity_type,
  bt.tour_category,
  bt.date_of_creation,
  bt.date_of_first_online,
  
  af.currency_iso_code,
  af.pricing_dates_covered_next_12mo,
  af.pricing_valid_from,
  af.pricing_valid_to,
  
  af.num_individual_categories,
  af.num_group_categories,
  af.num_addons,
  
  CASE WHEN af.num_addons > 0 THEN 1 ELSE 0 END AS has_addons_configured,
  CASE WHEN af.num_individual_categories > 0 THEN 1 ELSE 0 END AS has_individual_pricing,
  CASE WHEN af.num_group_categories > 0 THEN 1 ELSE 0 END AS has_group_pricing,
  
  af.total_individual_price_tiers,
  af.total_group_price_tiers,
  af.total_addon_price_tiers,
  (af.total_individual_price_tiers + af.total_group_price_tiers + af.total_addon_price_tiers) AS total_price_tiers,
  
  af.has_volume_pricing_individual,
  af.has_volume_pricing_group,
  af.has_volume_pricing_addons,
  
  CASE 
    WHEN af.has_volume_pricing_individual = 1 
      OR af.has_volume_pricing_group = 1 
      OR af.has_volume_pricing_addons = 1 
    THEN 1 ELSE 0 
  END AS has_any_volume_pricing,
  
  af.config_snapshot_timestamp

FROM base_tours bt
INNER JOIN aggregated_features af ON bt.tour_option_id = af.tour_option_id

ORDER BY bt.tour_id, bt.tour_option_id
